In [1]:
from datasets import load_dataset

ds_2024 = load_dataset("Reubencf/2024_events")
ds_2025 = load_dataset("Reubencf/2025_events")

from datasets import concatenate_datasets

combined_events = concatenate_datasets([
    ds_2024["train"],
    ds_2025["train"]
])

events = combined_events.map(
    lambda x: {"section": x["section"].lower()}
)

In [32]:
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from langchain_core.documents import Document
from pprint import pprint
import re

# https://reference.langchain.com/python/langchain-chroma/vectorstores/Chroma

llm = ChatOllama(model="llama3.2:3b")
                 
event_documents = []
event_ids = []
section_documents = []
section_ids = []

events_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="events",
    embedding_function=OllamaEmbeddings(model="nomic-embed-text")
)

sections_vector_store = Chroma(
    persist_directory="./chroma_db",
    collection_name="sections",
    embedding_function=OllamaEmbeddings(model="nomic-embed-text")
)

ei = 0
si = 0

print("Processing events and generating summaries of sections...")

for event in events:
    date_prefix = f"{event['month']} {event['day']}, {event['year']}: "
    text = f"{date_prefix}{event['instruction']}\n\n{event['content']}"

    parts = re.split(r'(### .+)', event["response"])

    event_section_ids = []
    current_title = "Introduction"
    current_content = ""

    for part in parts:
        if part.startswith("###"):
            current_title = part.replace("###", "").strip()
            current_title = current_title.replace("**", "").strip()
            
            current_content = ""
        else:
            current_content = part
            current_content = current_content.replace("\n\n#", "").strip()
            current_content = current_content.replace("\n*", "").strip()
            current_content = current_content.replace("**", "").strip()
            current_content = current_content.replace("\n\n---", "").strip()
            current_content = current_content.strip();

            prompt = ChatPromptTemplate.from_template(
                """
                Summarize the following text clearly and concisely into one sentence. Only output the summary and nothing else.

                {current_content}
                """
            )

            chain = prompt | llm | StrOutputParser()

            summary = chain.invoke({"current_content": current_content})

            section_documents.append(Document(
                page_content = f"{current_title}: {current_content}", 
                metadata={
                    "event_id": f"e{ei}",
                    "title": current_title,
                    "summary": summary
                })
            )
            
            section_ids.append(f"s{si}")
            event_section_ids.append(f"s{si}")

            # if ei < 10:
            #     print("\n\nSection:")
            #     print((si, current_title, current_content, summary)) # print the last element
            
            si = si + 1


    event_documents.append(Document(
        page_content = text, # this will be converted to document
        metadata={
            "month": event["month"],
            "day": event["day"],
            "year": event["year"],
            "category": event["section"],
            "sections": event_section_ids
        })
    )
    
    event_ids.append(f"e{ei}")
    ei=ei+1
    
    print(f"\r{ei} / {len(events)} [ {round((ei) / len(events) * 100, 2)}% ]", end="")

    # if ei > 10:
    #     break

print("\nDone")

print("\nGenerating embeddings and storing...")
events_vector_store.reset_collection()
batch_size = 100

for i in range(0, len(event_documents), batch_size):
    doc_batch = event_documents[i:i+batch_size]
    id_batch = event_ids[i:i+batch_size]
    
    events_vector_store.add_documents(ids=id_batch, documents=doc_batch)

    print(f"\r{i + len(doc_batch)} / {len(event_documents)} [ {round((i + len(doc_batch)) / len(event_documents) * 100)}% ]", end="")

print("\nDone")



print("\nStoring sections...")
for i in range(0, len(section_documents), batch_size):
    doc_batch = section_documents[i:i+batch_size]
    id_batch = section_ids[i:i+batch_size]
    
    sections_vector_store.add_documents(ids=id_batch, documents=doc_batch)

    print(f"\r{i + len(doc_batch)} / {len(section_documents)} [ {round((i + len(doc_batch)) / len(section_documents) * 100)}% ]", end="")

print("\nDone!")





Processing events and generating summaries of sections...
11 / 10712 [ 0.1% ]Generating embeddings and storing...
11 / 11 [ 100% ]
Done

Storing sections...
104 / 104 [ 100% ]
Done!


In [31]:
results = events_vector_store.get(limit=3)
pprint(results)

results = sections_vector_store.get(limit=20)
pprint(results)

{'data': None,
 'documents': ['January 1, 2024: \n'
               '\n'
               'Instruction: Following the withdrawal of five brigades from '
               'the Gaza Strip on January 1, 2024, what did the IDF mean by '
               'announcing a "different mode of operations" for the '
               'Israel-Hamas war?\n'
               '\n'
               'Content: Israel–Hamas war\n'
               '2023 Israeli invasion of the Gaza Strip\n'
               'The IDF withdraws five brigades, consisting of thousands of '
               'soldiers, from the Gaza Strip and says the war will enter a '
               '"different mode of operations". (Al Jazeera)',
               'January 1, 2024: \n'
               '\n'
               'Instruction: What was the impact of Hamas firing at least 27 '
               'rockets at cities and towns in central and southern Israel '
               'shortly after midnight on January 1, 2024?\n'
               '\n'
               'Content: Is